In [2]:
# Cell: Load and Predict with Confidence
import numpy as np
import joblib
import tensorflow as tf

# Load artifacts
scaler_loaded = joblib.load("../models/scaler.pkl")
feature_cols_loaded = joblib.load("../models/feature_cols.pkl")
model_loaded = tf.keras.models.load_model("../models/malware_dnn.h5")

# Label encoder is optional (if multi-class or string labels)
try:
    le_loaded = joblib.load("label_encoder.pkl")
except:
    le_loaded = None

def predict_with_confidence(sample_df):
    """
    sample_df: pandas DataFrame with ONE or MORE rows of raw features (same columns as training)
    """
    # Ensure correct column order and missing columns handling
    for col in feature_cols_loaded:
        if col not in sample_df.columns:
            sample_df[col] = 0
    sample_df = sample_df[feature_cols_loaded]

    # Scale
    x_scaled = scaler_loaded.transform(sample_df)

    # Predict
    probs = model_loaded.predict(x_scaled)

    # Binary case: shape (n,1) sigmoid
    if probs.shape[1] == 1:
        conf = float(probs[0][0])
        pred_class = 1 if conf >= 0.5 else 0
        confidence = conf if pred_class == 1 else (1 - conf)

        label = "Malware" if pred_class == 1 else "Benign"
        return label, confidence, conf  # (label, confidence, raw malware probability)

    # Multi-class case: shape (n,C) softmax
    else:
        pred_idx = int(np.argmax(probs[0]))
        confidence = float(np.max(probs[0]))

        if le_loaded is not None:
            label = str(le_loaded.inverse_transform([pred_idx])[0])
        else:
            label = f"Class_{pred_idx}"

        return label, confidence, probs[0].tolist()


In [3]:
# Example: take one row from test set (raw, before scaling)
import pandas as pd
df = pd.read_csv('../data/Malware dataset.csv')
X_test = df.drop(columns=['classification','hash'])

In [4]:
X_test.columns

Index(['millisecond', 'state', 'usage_counter', 'prio', 'static_prio',
       'normal_prio', 'policy', 'vm_pgoff', 'vm_truncate_count', 'task_size',
       'cached_hole_size', 'free_area_cache', 'mm_users', 'map_count',
       'hiwater_rss', 'total_vm', 'shared_vm', 'exec_vm', 'reserved_vm',
       'nr_ptes', 'end_data', 'last_interval', 'nvcsw', 'nivcsw', 'min_flt',
       'maj_flt', 'fs_excl_counter', 'lock', 'utime', 'stime', 'gtime',
       'cgtime', 'signal_nvcsw'],
      dtype='object')

In [5]:
sample = X_test.iloc[[1000]].copy()

label, confidence, extra = predict_with_confidence(sample)

print("Prediction:", label)
print("Confidence:", round(confidence * 100, 2), "%")
print("Extra:", extra)  # malware probability (binary) OR full probability list (multi-class)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 445ms/step
Prediction: Benign
Confidence: 100.0 %
Extra: 8.269793738691078e-08


In [6]:
sample = X_test.iloc[[100]].copy()

label, confidence, extra = predict_with_confidence(sample)

print("Prediction:", label)
print("Confidence:", round(confidence * 100, 2), "%")
print("Extra:", extra)  # malware probability (binary) OR full probability list (multi-class)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step
Prediction: Malware
Confidence: 100.0 %
Extra: 1.0


In [7]:
# Example: take one row from test set (raw, before scaling)
import pandas as pd
df = pd.read_csv('../data/Malware dataset.csv')
X_test = df.drop(columns=['hash'])

In [8]:
X_test.columns

Index(['millisecond', 'classification', 'state', 'usage_counter', 'prio',
       'static_prio', 'normal_prio', 'policy', 'vm_pgoff', 'vm_truncate_count',
       'task_size', 'cached_hole_size', 'free_area_cache', 'mm_users',
       'map_count', 'hiwater_rss', 'total_vm', 'shared_vm', 'exec_vm',
       'reserved_vm', 'nr_ptes', 'end_data', 'last_interval', 'nvcsw',
       'nivcsw', 'min_flt', 'maj_flt', 'fs_excl_counter', 'lock', 'utime',
       'stime', 'gtime', 'cgtime', 'signal_nvcsw'],
      dtype='object')

In [9]:
X_test['classification'].value_counts()

classification
malware    50000
benign     50000
Name: count, dtype: int64

In [10]:
# Select 10 benign
benign_samples = X_test[X_test["classification"] == 'malware'].head(10)

# Select 10 malware
malware_samples = X_test[X_test["classification"] == 'benign'].head(10)

# Combine
val_df = pd.concat([benign_samples, malware_samples])

# Separate features only (remove target)
X_val = val_df.drop(columns=["classification"])

In [11]:
X_val.to_csv("../data/Xvalidation.csv", index=False)